# Eğitici LLM – küçük SFT testi (Qwen3.5-4B, LoRA) — v2

Bu notebook şu adımları izler:
1. Modeli yükler ve **eğitimden önce** 12 test sorusuna cevap ürettirir (düşünme modu açık).
2. `sft_egitim.jsonl` verisiyle LoRA SFT eğitimi yapar.
3. Aynı 12 soruyu **eğitimden sonra** tekrar sorar (düşünme modu açık, aynı token sınırı).
4. İki sonucu yan yana `comparison.md` dosyasına yazar.

Her değerlendirme ve eğitim adımı şunları raporlar: **süre**, **üretilen/işlenen toplam token sayısı** ve **hız (token/saniye)**.

**Kaggle ayarları (sağ panel):** Accelerator = GPU T4 x2 (yalnızca bir GPU kullanılır), Internet = açık.
`sft_egitim.jsonl` ve `degerlendirme_istemleri.jsonl` dosyalarını bir Kaggle veri seti olarak notebook'a ekle (Add Input).

Uzun çalıştırma için **Save Version → Save & Run All** kullan. Çıktılar `/kaggle/working` altına kaydedilir ve sürümün **Output** sekmesinde görünür.

## 1. Kurulum
Unsloth kütüphanesini kurar. Unsloth, ince ayarı daha hızlı ve daha az bellekle yapmamızı sağlar.

In [ ]:
!pip install -q unsloth   # Unsloth'u ve bağımlılıklarını (transformers, peft, trl vb.) sessiz modda kurar

## 2. Ayarlar ve dosyalar
Bütün ayarlar bu hücrede tek yerde tanımlanır. Bir değeri değiştirmek istersen yalnızca burayı düzenlemen yeterli.

In [ ]:
import os                                    # İşletim sistemi ayarları ve dosya yolları için
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # T4 x2'de sadece ilk GPU'yu görünür yapar; Unsloth çoklu GPU ile çalışmaz

from unsloth import FastLanguageModel        # Unsloth'un model yükleyicisi; transformers'tan ÖNCE import edilmeli
import glob                                  # Dosya aramak için (joker karakterli yol eşleştirme)
import json                                  # JSON / JSONL dosyalarını okuyup yazmak için
import time                                  # Süre ölçmek için
import torch                                 # PyTorch: tensörler, GPU işlemleri

MODEL_NAME = "unsloth/Qwen3.5-4B"            # Eğitilecek temel model (Hugging Face'teki adı)
MAX_SEQ_LEN = 2048                           # Bir eğitim örneğinin en fazla token uzunluğu
EPOCHS = 3                                   # Eğitim verisinin üzerinden kaç tur geçileceği
LEARNING_RATE = 2e-4                         # Öğrenme oranı: her adımda ağırlıkların ne kadar değişeceği
LORA_RANK = 16                               # LoRA adaptör matrislerinin genişliği (kapasite)
MAX_NEW_TOKENS_EVAL = 1536                   # Değerlendirmede bir cevap için üretilebilecek en fazla token
OUT_DIR = "/kaggle/working"                  # Çıktı dosyalarının yazılacağı klasör (Kaggle'da kalıcı olan yer)

def find_file(name):                                                    # Veri setindeki bir dosyayı adıyla bulan yardımcı fonksiyon
    paths = glob.glob(f"/kaggle/input/**/{name}", recursive=True)       # /kaggle/input altındaki bütün klasörlerde arar
    assert paths, f"{name} not found. Add the dataset with Add Input."  # Bulunamazsa açıklayıcı bir hata verir
    return paths[0]                                                     # Bulunan ilk yolu döndürür

TRAIN_PATH = find_file("sft_egitim.jsonl")                # Eğitim verisinin yolu
EVAL_PATH = find_file("degerlendirme_istemleri.jsonl")    # Test sorularının yolu

gpu = torch.cuda.get_device_properties(0)                 # GPU'nun özelliklerini okur
print(gpu.name, f"{gpu.total_memory / 1e9:.1f} GB")       # GPU adını ve belleğini GB cinsinden yazdırır

## 3. Modeli yükleme
Model 16-bit olarak yüklenir. 4B model yaklaşık 10 GB bellek kullanır ve tek bir T4'e (16 GB) sığar.
Unsloth, Qwen3.5 mimarisi için 4-bit QLoRA önermediği için 16-bit LoRA kullanıyoruz.

In [ ]:
start_time = time.time()                                  # Yükleme süresini ölçmek için kronometreyi başlatır

model, tokenizer = FastLanguageModel.from_pretrained(     # Modeli ve tokenizer'ı indirip belleğe yükler
    model_name=MODEL_NAME,                                # Hangi model
    max_seq_length=MAX_SEQ_LEN,                           # Desteklenecek en uzun dizi
    load_in_4bit=False,                                   # 4-bit sıkıştırma kapalı
    load_in_16bit=True,                                   # 16-bit yükleme (T4'te otomatik olarak fp16 olur)
    full_finetuning=False,                                # Tam ince ayar değil, LoRA kullanacağız
)

tok = getattr(tokenizer, "tokenizer", tokenizer)          # Qwen3.5 görsel-dil modeli; tokenizer bir "processor" içinde gelebilir, metin tokenizer'ını alır
if getattr(tok, "chat_template", None) is None:           # Metin tokenizer'ında sohbet şablonu yoksa
    tok.chat_template = tokenizer.chat_template           # processor'daki şablonu ona kopyalar
if tok.pad_token is None:                                 # Dolgu (padding) tokeni tanımlı değilse
    tok.pad_token = tok.eos_token                         # bitiş tokenini dolgu tokeni olarak kullanır

print(f"Model loaded in {time.time() - start_time:.0f} s")  # Yükleme süresini yazdırır

## 4. Veri hazırlığı

Qwen'in sohbet şablonu, son kullanıcı mesajından **önceki** asistan turlarındaki düşünme izini siler.
Bu yüzden çok turlu her diyalog, içindeki asistan turu sayısı kadar ayrı örneğe bölünür.
Her örnekte loss (hata) **yalnızca son asistan turu** üzerinden hesaplanır: düşünme izi + cevap.
Böylece model eğitimde, kullanımda göreceği formatın aynısını görür.

Hücrenin sonundaki kontrolde **LOSS PART** bölümü yalnızca düşünme izi ve asistan cevabını içermeli; kullanıcı mesajı orada görünmemeli.

In [ ]:
def read_jsonl(path):                                                       # Bir JSONL dosyasını okuyan fonksiyon
    with open(path, encoding="utf-8") as f:                                 # Dosyayı UTF-8 (Türkçe karakter desteği) ile açar
        return [json.loads(line) for line in f if line.strip()]             # Boş olmayan her satırı bir sözlüğe çevirip liste yapar

train_raw = read_jsonl(TRAIN_PATH)                                          # Eğitim diyalogları
eval_raw = read_jsonl(EVAL_PATH)                                            # Test soruları

def build_examples(messages):                                               # Bir diyalogdan eğitim örnekleri üreten fonksiyon
    examples = []                                                           # Üretilen örnekler bu listeye eklenecek
    for i, message in enumerate(messages):                                  # Diyalogdaki her mesajı sırasıyla (indeksiyle) gezer
        if message["role"] != "assistant":                                  # Asistan mesajı değilse
            continue                                                        # atlar; örnekler yalnızca asistan turlarında biter
        prefix = tok.apply_chat_template(                                   # Asistan cevabından ÖNCEKİ kısmı şablona çevirir
            messages[:i], tokenize=False, add_generation_prompt=True        # add_generation_prompt: sonuna "<|im_start|>assistant" ekler
        )
        full = tok.apply_chat_template(messages[:i + 1], tokenize=False)    # Asistan cevabı DAHİL tam metni şablona çevirir
        if full.startswith(prefix):                                         # Tam metin, öncesiyle birebir başlıyorsa
            cut = len(prefix)                                               # loss'un başlayacağı karakter konumu = önceki kısmın uzunluğu
        else:                                                               # Şablon farkı olursa (beklenmeyen durum)
            cut = full.rfind("<|im_start|>assistant\n")                     # son asistan başlığını bulur
            assert cut != -1, "Unexpected chat template output"             # bulunamazsa hata verir
            cut += len("<|im_start|>assistant\n")                           # başlıktan hemen sonrasını başlangıç alır
        encoded = tok(full, add_special_tokens=False, return_offsets_mapping=True)   # Tam metni token'lara çevirir; her tokenin metindeki konumunu da verir
        labels = [                                                          # Her token için hedef (label) listesi
            token_id if start >= cut else -100                              # cut'tan sonraki tokenlar hedef; öncekiler -100 (loss'a katılmaz)
            for token_id, (start, end) in zip(encoded["input_ids"], encoded["offset_mapping"])
        ]
        examples.append({                                                   # Örneği listeye ekler
            "input_ids": encoded["input_ids"],                              # Modele verilecek token dizisi
            "attention_mask": encoded["attention_mask"],                    # Hangi tokenların gerçek olduğunu gösteren maske
            "labels": labels,                                               # Öğrenilecek hedefler
        })
    return examples                                                         # Bu diyalogdan çıkan bütün örnekleri döndürür

examples = [ex for dialog in train_raw for ex in build_examples(dialog["messages"])]   # Bütün diyaloglardan örnekleri tek listede toplar
examples = [ex for ex in examples if len(ex["input_ids"]) <= MAX_SEQ_LEN]              # Çok uzun örnekleri çıkarır
lengths = [len(ex["input_ids"]) for ex in examples]                                    # Her örneğin token uzunluğu
train_tokens_per_epoch = sum(lengths)                                                  # Bir epoch'ta işlenen toplam token

print(f"{len(train_raw)} dialogs -> {len(examples)} training examples")                # Diyalog ve örnek sayısı
print(f"avg {sum(lengths) / len(lengths):.0f} tokens, max {max(lengths)} tokens, "
      f"{train_tokens_per_epoch} tokens per epoch")                                    # Ortalama/en uzun örnek ve epoch başına token

sample = examples[1]                                                                   # Kontrol için ikinci örneği seçer
print("=== FULL TEXT ===\n" + tok.decode(sample["input_ids"]))                         # Modele verilen metnin tamamı
print("=== LOSS PART ===\n" + tok.decode(                                              # Loss'un hesaplandığı kısım
    [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]            # sadece label'ı -100 olmayan tokenlar
))

## 5. Cevap üretme ve değerlendirme fonksiyonları

- `generate_reply`: **tek bir soruyu** modele sorar; düşünme izini, cevabı ve üretilen token sayısını döndürür.
- `evaluate`: **12 test sorusunun hepsini** sırayla `generate_reply` ile sorar; toplam süreyi, toplam token sayısını ve hızı (token/saniye) hesaplar, sonuçları JSON dosyasına kaydeder.

Test soruları eğitim verisinde **yok**. Böylece modelin davranışı yeni sorulara genelleyip genellemediğine bakıyoruz.

In [ ]:
def generate_reply(messages, max_new_tokens, thinking=True):                # Tek bir soruya cevap üreten fonksiyon
    prompt_text = tok.apply_chat_template(                                  # Mesajları Qwen sohbet şablonuna çevirir
        messages,                                                           # Sohbet geçmişi ve soru
        tokenize=False,                                                     # Sonucu metin olarak ver (token değil)
        add_generation_prompt=True,                                         # Sonuna "<|im_start|>assistant" ekle: sıra modelde
        enable_thinking=thinking,                                           # Düşünme modunu aç/kapat
    )
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")   # Metni token'a çevirip GPU'ya taşır
    input_len = inputs["input_ids"].shape[1]                                # Girdinin token uzunluğu

    with torch.no_grad():                                                   # Gradyan hesaplama kapalı: bellek ve hız kazancı
        output = model.generate(                                            # Modelin cevap ürettiği çağrı
            **inputs,                                                       # Token dizisi ve attention mask
            max_new_tokens=max_new_tokens,                                  # En fazla bu kadar yeni token üret
            do_sample=True,                                                 # Olasılıklara göre örnekle (en olası tokeni seçmek yerine)
            temperature=0.7,                                                # Rastgelelik düzeyi
            top_p=0.8,                                                      # Olasılık toplamı %80'e ulaşan tokenlar arasından seç
            top_k=20,                                                       # En olası 20 token arasından seç
        )

    n_new_tokens = output.shape[1] - input_len                              # Üretilen token sayısı = toplam uzunluk - girdi uzunluğu
    text = tok.decode(output[0][input_len:], skip_special_tokens=True)      # Sadece yeni tokenları metne çevirir

    if "</think>" in text:                                                  # Model düşünmeyi bitirip cevaba geçtiyse
        reasoning, response = text.split("</think>", 1)                     # Metni düşünme ve cevap olarak ikiye böler
        return reasoning.strip(), response.strip(), n_new_tokens            # Düşünme, cevap ve token sayısını döndürür
    if not thinking:                                                        # Düşünme kapalıysa metnin tamamı cevaptır
        return "", text.strip(), n_new_tokens
    return text.strip(), "[TRUNCATED: token limit reached while thinking]", n_new_tokens   # Düşünme sınıra takılıp yarım kaldı


def evaluate(label, thinking=True, max_new_tokens=MAX_NEW_TOKENS_EVAL):     # Bütün test sorularını soran fonksiyon
    FastLanguageModel.for_inference(model)                                  # Modeli cevap üretme moduna alır (daha hızlı)
    torch.manual_seed(0)                                                    # Rastgeleliği sabitler: tekrarlanabilir sonuçlar

    results = []                                                            # Her sorunun sonucu bu listeye eklenecek
    total_tokens = 0                                                        # Üretilen toplam token sayacı
    truncated = 0                                                           # Token sınırına takılan cevap sayacı
    start_time = time.time()                                                # Kronometreyi başlatır

    for item in eval_raw:                                                   # Her test sorusu için
        question_start = time.time()                                        # Bu sorunun süresini ölçmeye başlar
        reasoning, response, n_tokens = generate_reply(                     # Soruyu modele sorar, üç değeri alır
            item["messages"], max_new_tokens, thinking
        )
        question_time = time.time() - question_start                        # Bu sorunun süresi
        total_tokens += n_tokens                                            # Token sayısını toplama ekler
        if n_tokens >= max_new_tokens:                                      # Sınıra kadar ürettiyse
            truncated += 1                                                  # kesik cevap sayacını artırır

        results.append({                                                    # Bu sorunun bilgilerini listeye ekler
            "id": item["id"],                                               # Soru kimliği (T01, T02, ...)
            "category": item["kategori"],                                   # Kategori (anahtar veri dosyasından geldiği için Türkçe)
            "prompt": item["messages"][-1]["content"],                      # Son kullanıcı mesajı, yani soru
            "response": response,                                           # Modelin cevabı
            "reasoning": reasoning,                                         # Modelin düşünme izi
            "n_tokens": n_tokens,                                           # Bu cevapta üretilen token sayısı
            "seconds": round(question_time, 1),                             # Bu cevabın süresi
        })
        print(f"[{item['id']}] {n_tokens} tokens, {question_time:.0f} s | "
              f"{item['messages'][-1]['content'][:60]}")                    # Soru numarası, token, süre ve sorunun başı
        print(f"-> {response[:300]}\n")                                     # Cevabın ilk 300 karakteri

    elapsed = time.time() - start_time                                      # Toplam süre
    speed = total_tokens / elapsed                                          # Hız: toplam token / toplam süre
    print("=" * 60)                                                         # Ayırıcı çizgi
    print(f"{label} | time: {elapsed:.0f} s | total tokens: {total_tokens} | "
          f"speed: {speed:.1f} tokens/s | truncated: {truncated}/{len(eval_raw)}")   # Özet satırı

    summary = {"label": label, "seconds": round(elapsed, 1), "total_tokens": total_tokens,
               "tokens_per_second": round(speed, 2), "truncated": truncated}       # Özet bilgileri bir sözlükte toplar
    with open(f"{OUT_DIR}/results_{label}.json", "w", encoding="utf-8") as f:     # Sonuç dosyasını açar (with: iş bitince kapatır)
        json.dump({"summary": summary, "results": results}, f, ensure_ascii=False, indent=1)   # Özet ve sonuçları kaydeder
    return summary, results                                                 # Özeti ve sonuç listesini döndürür

## 6. Eğitim öncesi değerlendirme
Eğitilmemiş model, düşünme modu açık olarak test edilir.
En kötü durumda (bütün cevaplar sınıra takılırsa) süre ≈ 12 × 1536 / hız. Hız ~10 token/sn ise yaklaşık 30 dakika.

In [ ]:
before_summary, before = evaluate("before", thinking=True)    # Eğitim öncesi testi çalıştırır; özeti ve sonuçları saklar

## 7. LoRA ile SFT eğitimi
Ana model ağırlıkları dondurulur; yalnızca eklenen küçük LoRA matrisleri eğitilir.
Eğitim sonunda süre, işlenen toplam token, hız (token/saniye), en yüksek GPU belleği ve loss değerleri yazdırılır.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq   # Hugging Face eğitim araçları
from datasets import Dataset                                                  # Python listesini eğitim veri setine çevirmek için

model = FastLanguageModel.get_peft_model(        # Modele LoRA adaptörlerini ekler
    model,                                       # Temel model
    r=LORA_RANK,                                 # Adaptör matrislerinin genişliği
    target_modules=[                             # LoRA'nın ekleneceği katmanlar
        "q_proj", "k_proj", "v_proj", "o_proj",  # Dikkat (attention) katmanları
        "gate_proj", "up_proj", "down_proj",     # MLP katmanları
    ],
    lora_alpha=LORA_RANK,                        # Adaptör etkisinin ölçeği
    lora_dropout=0,                              # Dropout yok (Unsloth'un önerdiği hızlı ayar)
    bias="none",                                 # Bias parametreleri eğitilmez
    use_gradient_checkpointing="unsloth",        # Bellek tasarrufu: bazı ara sonuçları saklamak yerine yeniden hesaplar
    random_state=3407,                           # Tekrarlanabilirlik için sabit tohum
    max_seq_length=MAX_SEQ_LEN,                  # En uzun dizi
)
FastLanguageModel.for_training(model)            # Modeli tekrar eğitim moduna alır (değerlendirmede inference moduna geçmişti)

use_bf16 = torch.cuda.is_bf16_supported()        # GPU bf16 destekliyor mu? (T4: hayır)

args = TrainingArguments(                        # Eğitim ayarları
    output_dir=f"{OUT_DIR}/checkpoints",         # Ara dosyaların klasörü
    per_device_train_batch_size=1,               # Her adımda GPU'ya 1 örnek
    gradient_accumulation_steps=4,               # 4 örneğin gradyanını biriktirip tek güncelleme yapar (etkin batch = 4)
    num_train_epochs=EPOCHS,                     # Kaç epoch
    learning_rate=LEARNING_RATE,                 # Öğrenme oranı
    warmup_steps=5,                              # İlk 5 adımda öğrenme oranını yavaşça artırır
    lr_scheduler_type="linear",                  # Sonra öğrenme oranını doğrusal olarak azaltır
    weight_decay=0.01,                           # Aşırı öğrenmeye karşı hafif düzenleme
    optim="adamw_8bit",                          # 8-bit AdamW optimizer: daha az bellek
    logging_steps=5,                             # Her 5 adımda bir loss kaydeder
    bf16=use_bf16,                               # Destekleniyorsa bf16
    fp16=not use_bf16,                           # Desteklenmiyorsa fp16 (T4)
    save_strategy="no",                          # Ara checkpoint kaydetme (adaptörü sonda kaydedeceğiz)
    report_to="none",                            # Harici izleme servislerine gönderme
    remove_unused_columns=False,                 # Kendi hazırladığımız sütunları silme
    seed=3407,                                   # Tekrarlanabilirlik
)

trainer = Trainer(                                                         # Eğitimi yürütecek nesne
    model=model,                                                           # LoRA eklenmiş model
    args=args,                                                             # Yukarıdaki ayarlar
    train_dataset=Dataset.from_list(examples).shuffle(seed=3407),          # Örnekleri veri setine çevirip karıştırır
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),   # Batch oluştururken dolgu ekler; dolgu loss'a katılmaz
)

torch.cuda.reset_peak_memory_stats()             # En yüksek bellek ölçümünü sıfırlar
start_time = time.time()                         # Eğitim kronometresini başlatır
train_stats = trainer.train()                    # EĞİTİM
train_time = time.time() - start_time            # Eğitim süresi

train_total_tokens = train_tokens_per_epoch * EPOCHS        # İşlenen toplam token = epoch başına token × epoch
train_speed = train_total_tokens / train_time               # Eğitim hızı (token/saniye)
peak_memory = torch.cuda.max_memory_allocated() / 1e9       # Eğitim boyunca kullanılan en yüksek GPU belleği (GB)

print("=" * 60)
print(f"training | time: {train_time / 60:.1f} min | steps: {train_stats.global_step} | "
      f"avg loss: {train_stats.training_loss:.3f}")                                    # Süre, adım sayısı, ortalama loss
print(f"training | total tokens: {train_total_tokens} | speed: {train_speed:.0f} tokens/s | "
      f"peak GPU memory: {peak_memory:.1f} GB")                                        # Token, hız, bellek
loss_history = [(h["step"], round(h["loss"], 3)) for h in trainer.state.log_history if "loss" in h]   # Kaydedilen loss değerleri
print("loss by step:", loss_history)                                                   # Adım adım loss

## 8. Eğitim sonrası değerlendirme
Eğitilmiş model **aynı ayarlarla** (düşünme açık, aynı token sınırı) test edilir. Böylece önce/sonra farkı ayardan değil, eğitimden kaynaklanır.

In [ ]:
after_summary, after = evaluate("after", thinking=True)    # Eğitim sonrası testi çalıştırır

## 9. Karşılaştırma
Önce/sonra özetini ve her sorunun token sayısını tablo olarak yazdırır; bütün cevapları yan yana `comparison.md` dosyasına kaydeder.

In [ ]:
print(f"{'':8}{'time (s)':>10}{'tokens':>10}{'tok/s':>8}{'truncated':>11}")   # Özet tablosunun başlığı
for s in (before_summary, after_summary):                                        # Önce ve sonra özetleri için
    print(f"{s['label']:8}{s['seconds']:>10.0f}{s['total_tokens']:>10}"
          f"{s['tokens_per_second']:>8.1f}{s['truncated']:>11}")                  # Bir satır yazdırır
print()
print(f"{'id':6}{'before':>8}{'after':>8}  category")                             # Soru bazında token tablosu başlığı
for b, a in zip(before, after):                                                   # Aynı sorunun önce/sonra sonuçlarını eşleştirir
    print(f"{b['id']:6}{b['n_tokens']:>8}{a['n_tokens']:>8}  {b['category']}")    # Soru başına token sayıları

lines = ["# Before / after comparison\n"]                                         # Markdown dosyasının satırları
for s in (before_summary, after_summary):                                         # Özetleri dosyanın başına ekler
    lines.append(f"- **{s['label']}**: {s['seconds']:.0f} s, {s['total_tokens']} tokens, "
                 f"{s['tokens_per_second']:.1f} tokens/s, truncated {s['truncated']}/{len(before)}")
lines.append("")
for b, a in zip(before, after):                                                   # Her soru için
    lines += [
        f"## {b['id']} ({b['category']})",                                        # Başlık: soru kimliği ve kategorisi
        f"**Prompt:** {b['prompt']}\n",                                           # Soru
        f"**Before** ({b['n_tokens']} tokens):\n",                                # Eğitim öncesi cevap başlığı
        b["response"] + "\n",                                                     # Eğitim öncesi cevap
        f"**After** ({a['n_tokens']} tokens):\n",                                 # Eğitim sonrası cevap başlığı
        a["response"] + "\n",                                                     # Eğitim sonrası cevap
        "---\n",                                                                  # Ayırıcı
    ]
with open(f"{OUT_DIR}/comparison.md", "w", encoding="utf-8") as f:                # Dosyayı açar
    f.write("\n".join(lines))                                                     # Satırları birleştirip yazar
print("\nSaved:", f"{OUT_DIR}/comparison.md")                                     # Kaydedildiğini bildirir

## 10. Adaptörü kaydetme
Eğitilmiş LoRA adaptörünü kaydeder ve kolay indirmek için zip dosyası oluşturur. Adaptör yalnızca eklenen küçük matrisleri içerir (birkaç yüz MB); kullanmak için temel modelle birlikte yüklenir.

In [ ]:
adapter_dir = f"{OUT_DIR}/lora_adapter"          # Adaptör klasörünün yolu
model.save_pretrained(adapter_dir)               # LoRA ağırlıklarını kaydeder
tok.save_pretrained(adapter_dir)                 # Tokenizer'ı (ve sohbet şablonunu) aynı klasöre kaydeder
!cd /kaggle/working && zip -qr lora_adapter.zip lora_adapter   # Klasörü indirmesi kolay olsun diye zip'ler
print(os.listdir(adapter_dir))                   # Kaydedilen dosyaları listeler